In [2]:
import zipfile

# Open the zip file
with zipfile.ZipFile('archive.zip', 'r') as zip_ref:
    # Extract everything into the current directory
    zip_ref.extractall('.')
    
print("Extraction complete!")

Extraction complete!


In [4]:
import tensorflow as tf
import pandas as pd
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [8]:
import tensorflow as tf
import pandas as pd
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

# --- CONFIGURATION ---
BASE_PATH = 'AcneDataset' 
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 32
# These must match your CSV column headers exactly
classes = ['Blackheads', 'Cyst', 'Papules', 'Pustules', 'Whiteheads']

def load_split(folder_name, csv_filename):
    folder_path = os.path.join(BASE_PATH, folder_name)
    csv_path = os.path.join(folder_path, csv_filename)
    
    # 1. Load CSV and clean column names
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    df['filename'] = df['filename'].astype(str)
    
    # 2. Setup Generator (Rescale pixel values to [0, 1])
    # Note: We add horizontal_flip for the training set only
    if folder_name == 'train':
        datagen = ImageDataGenerator(rescale=1./255, horizontal_flip=True)
    else:
        datagen = ImageDataGenerator(rescale=1./255)
    
    # 3. Flow from Dataframe
    return datagen.flow_from_dataframe(
        dataframe=df,
        directory=folder_path,
        x_col="filename",
        y_col=classes,
        target_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="raw",
        shuffle=(folder_name == 'train')
    )

# --- EXECUTION ---
print("Loading Training Set...")
train_generator = load_split('train', '_train_classes.csv')

print("\nLoading Validation Set...")
val_generator = load_split('valid', '_valid_classes.csv')

print("\nLoading Test Set...")
test_generator = load_split('test', '_test_classes.csv')

Loading Training Set...
Found 0 validated image filenames.

Loading Validation Set...
Found 0 validated image filenames.

Loading Test Set...
Found 0 validated image filenames.


C:\ProgramData\anaconda3\Lib\site-packages\keras\src\legacy\preprocessing\image.py:920: UserWarning: Found 2768 invalid image filename(s) in x_col="filename". These filename(s) will be ignored.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\keras\src\legacy\preprocessing\image.py:920: UserWarning: Found 921 invalid image filename(s) in x_col="filename". These filename(s) will be ignored.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\keras\src\legacy\preprocessing\image.py:920: UserWarning: Found 918 invalid image filename(s) in x_col="filename". These filename(s) will be ignored.
  warnings.warn(


In [9]:
import os
import pandas as pd

# 1. Check if the folder exists
base_path = 'AcneDataset'
train_path = os.path.join(base_path, 'train')

if not os.path.exists(train_path):
    print(f"❌ Error: The folder '{train_path}' was not found.")
else:
    print(f"✅ Folder found: {train_path}")
    
    # 2. Check the CSV
    csv_path = os.path.join(train_path, '_train_classes.csv')
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        csv_filename = df['filename'].iloc[0]
        print(f"📄 CSV found. First filename in CSV: '{csv_filename}'")
        
        # 3. Check the Folder
        folder_files = os.listdir(train_path)
        print(f"total files in folder: {len(folder_files)}")
        
        # 4. Try to find the match
        if csv_filename in folder_files:
            print(f"🚀 MATCH FOUND! The code should work.")
        else:
            print(f"❌ NO MATCH: The file '{csv_filename}' is not in the folder.")
            print(f"Sample of actual files in folder: {folder_files[:5]}")

✅ Folder found: AcneDataset\train
📄 CSV found. First filename in CSV: 'cystic_acne_-106-_jpg.rf.edd0354ffdf46368bdd005ba216c5fc3.jpg'
total files in folder: 6
❌ NO MATCH: The file 'cystic_acne_-106-_jpg.rf.edd0354ffdf46368bdd005ba216c5fc3.jpg' is not in the folder.
Sample of actual files in folder: ['Blackheads', 'Cyst', 'Papules', 'Pustules', 'Whiteheads']


In [10]:
import tensorflow as tf
import os

# 1. Configuration
BASE_PATH = 'AcneDataset'
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 32

# 2. Load the datasets directly from the directories
print("Loading Training Set...")
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_PATH, 'train'),
    labels='inferred',
    label_mode='categorical', # Use 'categorical' for multi-class
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=True
)

print("\nLoading Validation Set...")
val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_PATH, 'valid'),
    labels='inferred',
    label_mode='categorical',
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=True
)

print("\nLoading Test Set...")
test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_PATH, 'test'),
    labels='inferred',
    label_mode='categorical',
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=False
)

# 3. Check the class names
class_names = train_ds.class_names
print(f"\nSuccessfully loaded classes: {class_names}")

Loading Training Set...
Found 2778 files belonging to 5 classes.

Loading Validation Set...
Found 921 files belonging to 5 classes.

Loading Test Set...
Found 918 files belonging to 5 classes.

Successfully loaded classes: ['Blackheads', 'Cyst', 'Papules', 'Pustules', 'Whiteheads']


In [11]:
'''Upon closer inspection, we realized your images were already organized into subfolders by class (e.g., Blackheads, Cyst). 
This allowed us to use the more efficient image_dataset_from_directory method, which automatically assigns labels based on the folder names.'''

'''Training Set: 2,778 files across 5 classes.

Validation Set: 921 files across 5 classes.

Test Set: 918 files across 5 classes.

Categories: Confirmed the model is looking for Blackheads, Cyst, Papules, Pustules, and Whiteheads.'''

'Training Set: 2,778 files across 5 classes.\n\nValidation Set: 921 files across 5 classes.\n\nTest Set: 918 files across 5 classes.\n\nCategories: Confirmed the model is looking for Blackheads, Cyst, Papules, Pustules, and Whiteheads.'

In [12]:
import numpy as np

# Take one batch from your training data
for images, labels in train_ds.take(1):
    first_image = images[0].numpy()
    print("Min pixel value:", np.min(first_image))
    print("Max pixel value:", np.max(first_image))

Min pixel value: 0.0
Max pixel value: 255.0


In [13]:
'''Step 1: Normalization (The Math)
Neural networks struggle with large numbers like 255 because they cause the "gradients" (the way the model learns) to explode or become unstable.
We want to squash those values down to a range between 0 and 1.'''

'Step 1: Normalization (The Math)\nNeural networks struggle with large numbers like 255 because they cause the "gradients" (the way the model learns) to explode or become unstable.\nWe want to squash those values down to a range between 0 and 1.'

In [14]:
from tensorflow.keras import layers

# This layer will divide every pixel in your images by 255
normalization_layer = layers.Rescaling(1./255)

# We apply it to the dataset
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

print("Normalization complete. Pixels are now between 0 and 1.")

Normalization complete. Pixels are now between 0 and 1.


In [15]:
for images, labels in train_ds.take(1):
    first_image = images[0].numpy()
    print("New Min pixel value:", np.min(first_image))
    print("New Max pixel value:", np.max(first_image))

New Min pixel value: 0.08529412
New Max pixel value: 1.0


In [16]:
'''The Input & First Feature Extractors
This is where the model learns to see edges, redness, and textures.'''

'The Input & First Feature Extractors\nThis is where the model learns to see edges, redness, and textures.'

In [18]:
from tensorflow.keras import models, layers, regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import tensorflow as tf

def create_improved_cnn_model(input_shape=(256, 256, 3), num_classes=1):
    """
    Create an improved CNN model with better regularization and gradient handling
    """
    # Initialize sequential model
    model = models.Sequential()
    
    # 1. Input Layer
    model.add(layers.Input(shape=input_shape))
    
    # Optional: Add data augmentation layers (only active during training)
    model.add(layers.RandomFlip("horizontal_and_vertical"))
    model.add(layers.RandomRotation(0.1))
    model.add(layers.RandomZoom(0.1))
    model.add(layers.RandomContrast(0.1))
    
    # 2. First Convolutional Block
    # Use BatchNormalization after convolution, before activation (better gradient flow)
    model.add(layers.Conv2D(
        32, (3, 3), 
        padding='same',  # Preserve spatial dimensions
        kernel_regularizer=regularizers.l2(1e-4)  # L2 regularization
    ))
    model.add(layers.BatchNormalization())  # Normalize activations, helps with gradient flow
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.2))  # Dropout for regularization
    
    # 3. Second Convolutional Block
    model.add(layers.Conv2D(
        64, (3, 3), 
        padding='same',
        kernel_regularizer=regularizers.l2(1e-4)
    ))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.3))
    
    # 4. Third Convolutional Block (added for deeper feature extraction)
    model.add(layers.Conv2D(
        128, (3, 3), 
        padding='same',
        kernel_regularizer=regularizers.l2(1e-4)
    ))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.3))
    
    # 5. Fourth Convolutional Block
    model.add(layers.Conv2D(
        256, (3, 3), 
        padding='same',
        kernel_regularizer=regularizers.l2(1e-4)
    ))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.GlobalAveragePooling2D())  # Alternative to Flatten + Dense, reduces parameters
    model.add(layers.Dropout(0.4))
    
    # 6. Dense Layers with careful initialization
    # Use He initialization for ReLU activations
    model.add(layers.Dense(
        128, 
        kernel_initializer='he_normal',
        kernel_regularizer=regularizers.l2(1e-4)
    ))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Dropout(0.5))
    
    # 7. Output Layer
    if num_classes == 1:
        # Binary classification
        model.add(layers.Dense(1, activation='sigmoid'))
    else:
        # Multi-class classification
        model.add(layers.Dense(num_classes, activation='softmax'))
    
    return model


def get_optimizer(learning_rate=1e-3):
    """
    Get optimizer with gradient clipping to prevent exploding gradients
    """
    return Adam(
        learning_rate=learning_rate,
        beta_1=0.9,      # Momentum
        beta_2=0.999,    # RMSprop-like term
        epsilon=1e-7,    # Numerical stability
        clipnorm=1.0,    # Gradient clipping to prevent exploding gradients
    )

def get_callbacks():
    """
    Get training callbacks for better optimization and early stopping
    """
    callbacks = [
        # Reduce learning rate when validation loss plateaus
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,      # Reduce LR by half
            patience=5,      # Wait 5 epochs
            min_lr=1e-6,     # Minimum learning rate
            verbose=1
        ),
        
        # Stop training when validation loss doesn't improve
        EarlyStopping(
            monitor='val_loss',
            patience=15,      # Wait 15 epochs
            restore_best_weights=True,  # Restore best weights
            verbose=1
        ),
        
        # Optional: Model checkpoint
        # tf.keras.callbacks.ModelCheckpoint(
        #     'best_model.h5',
        #     monitor='val_loss',
        #     save_best_only=True,
        #     verbose=1
        # )
    ]
    
    return callbacks


def compile_model(model, num_classes=1):
    """
    Compile the model with appropriate loss and metrics
    """
    optimizer = get_optimizer()
    
    if num_classes == 1:
        loss = 'binary_crossentropy'
        metrics = [
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.AUC(name='auc')
        ]
    else:
        loss = 'categorical_crossentropy'
        metrics = ['accuracy']
    
    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=metrics
    )
    
    return model

# Usage example
if __name__ == "__main__":
    # Create model
    model = create_improved_cnn_model(input_shape=(256, 256, 3), num_classes=5)
    
    # Compile model
    model = compile_model(model, num_classes=5)
    
    # Get callbacks
    callbacks = get_callbacks()
    
    # Print model summary
    print("=" * 60)
    print("Improved CNN Model Summary:")
    print("=" * 60)
    model.summary()
    
    print("\n" + "=" * 60)
    print("Model features implemented to prevent issues:")
    print("=" * 60)
    print("1. ✓ BatchNormalization - Prevents vanishing/exploding gradients")
    print("2. ✓ Dropout - Reduces overfitting")
    print("3. ✓ L2 Regularization - Reduces overfitting")
    print("4. ✓ Gradient Clipping - Prevents exploding gradients")
    print("5. ✓ Data Augmentation - Reduces overfitting")
    print("6. ✓ Learning Rate Scheduling - Better optimization")
    print("7. ✓ Early Stopping - Prevents overfitting")
    print("8. ✓ He Initialization - Better weight initialization")
    print("9. ✓ Global Average Pooling - Reduces parameters")
    print("10.✓ Multiple Conv Blocks - Better feature extraction")
    print("=" * 60)

Improved CNN Model Summary:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ random_flip_1 (RandomFlip)           │ (None, 256, 256, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ random_rotation_1 (RandomRotation)   │ (None, 256, 256, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ random_zoom_1 (RandomZoom)           │ (None, 256, 256, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ random_contrast_1 (RandomContrast)   │ (None, 256, 256, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 256, 256, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_5                │ (None, 256, 256, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_5 (Activation)            │ (None, 256, 256, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 128, 128, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 128, 128, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (None, 128, 128, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_6                │ (None, 128, 128, 64)        │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_6 (Activation)            │ (None, 128, 128, 64)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_4 (MaxPooling2D)       │ (None, 64, 64, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_6 (Dropout)                  │ (None, 64, 64, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_6 (Conv2D)                    │ (None, 64, 64, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_7                │ (None, 64, 64, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_7 (Activation)            │ (None, 64, 64, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_5 (MaxPooling2D)       │ (None, 32, 32, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_7 (Dropout)                  │ (None, 32, 32, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_7 (Conv2D)                    │ (None, 32, 32, 256)         │         295,1

 Total params: 424,389 (1.62 MB)

 Trainable params: 423,173 (1.61 MB)

 Non-trainable params: 1,216 (4.75 KB)


Model features implemented to prevent issues:
1. ✓ BatchNormalization - Prevents vanishing/exploding gradients
2. ✓ Dropout - Reduces overfitting
3. ✓ L2 Regularization - Reduces overfitting
4. ✓ Gradient Clipping - Prevents exploding gradients
5. ✓ Data Augmentation - Reduces overfitting
6. ✓ Learning Rate Scheduling - Better optimization
7. ✓ Early Stopping - Prevents overfitting
8. ✓ He Initialization - Better weight initialization
9. ✓ Global Average Pooling - Reduces parameters
10.✓ Multiple Conv Blocks - Better feature extraction


In [19]:
'''1. Robustness for Real-World Photos
The Data Augmentation block at the beginning (RandomFlip, Rotation, Zoom, and Contrast) is vital for your "clicked images" goal. Real-time photos won't always have perfect lighting or be perfectly centered. This code teaches the model to recognize a "Cyst" or "Blackhead" even if the photo is dark or tilted.

2. Solving Training "Frustrations"
This code includes three "Safety Nets" that prevent common training failures:

BatchNormalization: In your previous step, we normalized the pixels to 0–1. BatchNormalization does this inside the model's layers. It keeps the math "stable" so the model doesn't stop learning halfway through.

GlobalAveragePooling2D: Instead of just flattening all the data into a giant list, this "summarizes" the features. It makes your model smaller and much faster to run on a website or mobile app.

Gradient Clipping: This prevents the math from "exploding" (becoming infinitely large), which sometimes happens when training on complex skin textures.

3. The "Smart" Training Strategy (Callbacks)
The get_callbacks() function is like having a co-pilot during training:

ReduceLROnPlateau: If the model stops improving, it "slows down" (reduces the learning rate) to try and find a better solution more carefully.

EarlyStopping: If the model is no longer getting smarter, it stops the training automatically. This saves you time and prevents Overfitting (where the model memorizes your specific training images).'''

'1. Robustness for Real-World Photos\nThe Data Augmentation block at the beginning (RandomFlip, Rotation, Zoom, and Contrast) is vital for your "clicked images" goal. Real-time photos won\'t always have perfect lighting or be perfectly centered. This code teaches the model to recognize a "Cyst" or "Blackhead" even if the photo is dark or tilted.\n\n2. Solving Training "Frustrations"\nThis code includes three "Safety Nets" that prevent common training failures:\n\nBatchNormalization: In your previous step, we normalized the pixels to 0–1. BatchNormalization does this inside the model\'s layers. It keeps the math "stable" so the model doesn\'t stop learning halfway through.\n\nGlobalAveragePooling2D: Instead of just flattening all the data into a giant list, this "summarizes" the features. It makes your model smaller and much faster to run on a website or mobile app.\n\nGradient Clipping: This prevents the math from "exploding" (becoming infinitely large), which sometimes happens when tr

In [ ]:
'''Since you have 2,778 training images and we set the batch_size to 32, the math works like this
:$$2,778 \text{ total images} \div 32 \text{ images per batch} \approx 87 \text{ batches}$$
What happens during these 87 steps?The model takes the first 32 images (Batch 1).
It makes a guess, checks the answer, and updates its "brain" (weights).It repeats this until it has done all 87 batches.
Once it hits 87/87, Epoch 1 is finished. It then starts over for Epoch 2, but shuffles the images so it doesn't just memorize the order.'''

'Since you have 2,778 training images and we set the batch_size to 32, the math works like this\n:$$2,778 \text{ total images} \\div 32 \text{ images per batch} \x07pprox 87 \text{ batches}$$\nWhat happens during these 87 steps?The model takes the first 32 images (Batch 1).\nIt makes a guess, checks the answer, and updates its "brain" (weights).It repeats this until it has done all 87 batches.\nOnce it hits 87/87, Epoch 1 is finished. It then starts over for Epoch 2, but shuffles the images so it doesn\'t just memorize the order.'

In [20]:
# We use the dataset objects (train_ds, val_ds) and the callbacks we defined earlier
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50, # We set 50, but EarlyStopping will likely stop it sooner to save time
    callbacks=callbacks
)

Epoch 1/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 760s 9s/step - accuracy: 0.2894 - loss: 1.8632 - val_accuracy: 0.2269 - val_loss: 1.8412 - learning_rate: 0.0010
Epoch 2/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 569s 7s/step - accuracy: 0.3063 - loss: 1.6867 - val_accuracy: 0.2410 - val_loss: 2.0272 - learning_rate: 0.0010
Epoch 3/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 2189s 25s/step - accuracy: 0.3387 - loss: 1.5982 - val_accuracy: 0.2638 - val_loss: 1.8769 - learning_rate: 0.0010
Epoch 4/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 543s 6s/step - accuracy: 0.3575 - loss: 1.5446 - val_accuracy: 0.2660 - val_loss: 1.7718 - learning_rate: 0.0010
Epoch 5/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 539s 6s/step - accuracy: 0.3611 - loss: 1.5052 - val_accuracy: 0.2866 - val_loss: 1.6854 - learning_rate: 0.0010
Epoch 6/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 537s 6s/step - accuracy: 0.3772 - loss: 1.4734 - val_accuracy: 0.2921 - val_loss: 1.6715 - learning_rate: 0.0010
Epoch 7/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 1734s 20s/step - accuracy: 0.4230 - loss: 1.3815 - val

In [22]:
# Run this to save your work
model.save('acne_classifier_v1.keras')
print("Model saved! You can now safely close Jupyter.")

Model saved! You can now safely close Jupyter.
